In [1]:
import os 
data_raw_path = '../Formula1/data/data_raw'
dirs = [x for x in os.listdir(data_raw_path) if not x.startswith('.')]

for dirc in dirs:
    dir_raw_path = data_raw_path + '/' + dirc
    #os.filepaths


In [2]:
data_raw_path = '../Formula-1/data/data_raw'

file_paths = [dirpath+'/'+filename for dirpath, dirnames, filenames in os.walk(data_raw_path) for filename in filenames ]
for dirpath, dirnames, filenames in os.walk(data_raw_path):
    print(f"Current Folder: {dirpath}")
    print(f"Subdirectories: {dirnames}")
    print(f"Files: {filenames}")
    print("-" * 20)
print(file_paths)

[]


In [6]:
from pyspark.sql.functions import col, when, lit, input_file_name, regexp_extract, length, first
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import shutil
import os

# 1. Initialize Spark Session
spark = (SparkSession
         .builder
         .appName('SparkDBApp')
         .getOrCreate())

# 2. Silence the noisy CSV schema mismatch warnings in your console logs
spark.sparkContext.setLogLevel("ERROR")

# 3. Create the database target if it does not exist
spark.sql("CREATE DATABASE IF NOT EXISTS formula1_EDA_race")

# Define the full array of CSV source targets
file_names = [
    'drivers', 'event', 'laps', 'race_control_messages',
    'results', 'session_info', 'session_status', 
    'track_status', 'weather_data'
]

# 4. Your robust file extraction function
def load_files(file_name):
    df = spark.read.csv(
        f"data/data_raw/*/session_*/{file_name}.csv",
        header=True,
        inferSchema=True
    )

    df = (
        df
        .withColumn('source_file', input_file_name())
        .withColumn(
            'year',
            regexp_extract(col('source_file'), r'data_raw/(\d{4})/', 1).cast('int')
        )
        .withColumn(
            'session_number',
            regexp_extract(col('source_file'), r'session_(\d+)', 1).cast('int')
        )
        .withColumn(
            'type_of_race',
            regexp_extract(col('source_file'), r'session_\d+_([^./]+)', 1).cast('string')
        )
    )

    # Safely drop '_c0' only if it exists in this specific table's schema
    if '_c0' in df.columns:
        df = df.drop('_c0')

    return df


# 5. Loop through every file name, load it, and save it dynamically
for name in file_names:
    print(f"Loading and processing data for: {name}...")
    
    try:
        # Load and process the files
        processed_df = load_files(name)
        
        # --- FIXED PIVOT LOGIC FOR THE EVENT TABLE ---
        if name == 'event':
            print("Detected event table. Pivoting key-value structure...")
            
            # 1. Reload the file without dropping '_c0' so we keep both columns
            raw_event_df = spark.read.csv(
                f"data/data_raw/*/session_*/{name}.csv",
                header=True,
                inferSchema=True
            )
            
            csv_cols = raw_event_df.columns
            if len(csv_cols) >= 2:
                # FIX: Extract strings from the list using indexing
                key_col = csv_cols[0]   # Extracts string (e.g., '_c0')
                val_col = csv_cols[1]   # Extracts string (e.g., '7')
                
                # 2. Re-apply your tracking metadata
                meta_df = (
                    raw_event_df
                    .withColumn('source_file', input_file_name())
                    .withColumn('year', regexp_extract(col('source_file'), r'data_raw/(\d{4})/', 1).cast('int'))
                    .withColumn('session_number', regexp_extract(col('source_file'), r'session_(\d+)', 1).cast('int'))
                    .withColumn('type_of_race', regexp_extract(col('source_file'), r'session_\d+_([^./]+)', 1).cast('string'))
                )
                
                meta_cols = ['source_file', 'year', 'session_number', 'type_of_race']
                
                # 3. Pivot the table using clear string references
                processed_df = (
                    meta_df
                    .groupBy(meta_cols)
                    .pivot(key_col) # Correctly receives a string parameter now
                    .agg(first(col(val_col)))
                    .select(
                        'source_file',
                        'year',
                        'session_number',
                        'type_of_race',
                        'RoundNumber',
                        'Location'
                    )
                )
            else:
                print(f"Warning: event table columns {csv_cols} don't match expected key-value layout.")
        # --------------------------------------------
        
        target_table = f"formula1_eda_race.{name}"
        
        # 1. Clear Spark metadata
        print(f"Dropping table if it exists: {target_table}...")
        spark.sql(f"DROP TABLE IF EXISTS {target_table}")
        
        # 2. Physically remove the existing folder location on disk
        target_path = f"/sfs/weka/scratch/aw2kc/Formula1/spark-warehouse/formula1_eda_race.db/{name}"
        if os.path.exists(target_path):
            print(f"Removing existing directory location: {target_path}...")
            shutil.rmtree(target_path)
        
        # 3. Write the fresh DataFrame as a brand-new table
        print(f"Saving to table: {target_table}...")
        processed_df.write.mode("overwrite").saveAsTable(target_table)
        print(f"Successfully saved to {target_table}!\n")
        
    except Exception as e:
        print(f"Error processing {name}: {e}\n")

print("All tables successfully processed and saved to formula1_eda!")


Loading and processing data for: drivers...
Dropping table if it exists: formula1_eda_race.drivers...
Saving to table: formula1_eda_race.drivers...
Successfully saved to formula1_eda_race.drivers!

Loading and processing data for: event...
Detected event table. Pivoting key-value structure...
Dropping table if it exists: formula1_eda_race.event...
Saving to table: formula1_eda_race.event...
Successfully saved to formula1_eda_race.event!

Loading and processing data for: laps...
Dropping table if it exists: formula1_eda_race.laps...
Saving to table: formula1_eda_race.laps...


Successfully saved to formula1_eda_race.laps!

Loading and processing data for: race_control_messages...
Dropping table if it exists: formula1_eda_race.race_control_messages...
Saving to table: formula1_eda_race.race_control_messages...
Successfully saved to formula1_eda_race.race_control_messages!

Loading and processing data for: results...
Dropping table if it exists: formula1_eda_race.results...
Saving to table: formula1_eda_race.results...
Successfully saved to formula1_eda_race.results!

Loading and processing data for: session_info...
Dropping table if it exists: formula1_eda_race.session_info...
Saving to table: formula1_eda_race.session_info...
Successfully saved to formula1_eda_race.session_info!

Loading and processing data for: session_status...
Dropping table if it exists: formula1_eda_race.session_status...
Saving to table: formula1_eda_race.session_status...
Successfully saved to formula1_eda_race.session_status!

Loading and processing data for: track_status...
Dropping

In [25]:
# We want to add onto results the average air temp,
# humidity, pressure, rainfall, tracktemp, wind direction, and wind speed

weather_averages = spark.sql("""
    SELECT 
        session_number, year,
        AVG(AirTemp) AS avg_air_temp, 
        AVG(Humidity) AS avg_humidity, 
        AVG(Pressure) AS avg_pressure, 
        MAX(Rainfall) AS rain, 
        AVG(TrackTemp) AS avg_track_temp, 
        AVG(WindDirection) AS avg_wind_direction, 
        AVG(WindSpeed) AS avg_wind_speed 
    FROM formula1_eda_race.weather_data
    where type_of_race = "R"
    GROUP BY session_number, year
""")

weather_averages.show(5)
# Create a dataframe that details the number of flags each race
flag_counts = spark.sql("""
    SELECT 
        year,
        session_number,
        COUNT(CASE WHEN Flag IS NOT NULL AND Flag != 'CLEAR' THEN 1 END) AS active_flag_count
    FROM formula1_eda_race.race_control_messages 
    where 
    GROUP BY year, session_number
""")

flag_counts.show(5)

#Check if events table was pivoted
race_name = spark.sql('SELECT * FROM formula1_eda_race.event where type_of_race="R"')
race_name.show(5)


+--------------+----+------------------+------------------+------------------+-----+------------------+------------------+------------------+
|session_number|year|      avg_air_temp|      avg_humidity|      avg_pressure| rain|    avg_track_temp|avg_wind_direction|    avg_wind_speed|
+--------------+----+------------------+------------------+------------------+-----+------------------+------------------+------------------+
|            11|2023| 28.80853658536585| 32.80487804878049| 988.4018292682914|false| 48.57926829268294|174.53658536585365|1.4408536585365836|
|            22|2023| 26.96282051282048|              51.0|1014.9275641025635|false| 33.46089743589744|284.93589743589746|1.7948717948717932|
|             8|2024|21.596500000000006|             63.99|1018.7434999999994|false|46.317999999999984|           160.895|            0.9795|
|            11|2021| 27.22670157068063| 68.61570680628276| 977.6308900523562| true| 37.82251308900524|104.59162303664921|0.5675392670157061|
|     

In [9]:
# ── Reusable timedelta-string -> seconds parser ───────────────────────────────
# Works on any column formatted like "0 days 00:01:23.456000". Only fills in a
# piece (days/minutes/seconds) if that piece's regex actually matched, so rows
# with no pit stop (nulls) correctly stay null all the way through.
def timedelta_str_to_seconds(colname):
    days = when(
        length(regexp_extract(col(colname), r"(\d+) days", 1)) > 0,
        regexp_extract(col(colname), r"(\d+) days", 1).cast("int")
    )
    minutes = when(
        length(regexp_extract(col(colname), r"days \d{2}:(\d{2}):", 1)) > 0,
        regexp_extract(col(colname), r"days \d{2}:(\d{2}):", 1).cast("int")
    )
    seconds = when(
        length(regexp_extract(col(colname), r"days \d{2}:\d{2}:(\d{2}\.\d+)", 1)) > 0,
        regexp_extract(col(colname), r"days \d{2}:\d{2}:(\d{2}\.\d+)", 1).cast("double")
    )
    # days * 86400 (seconds/day) + minutes * 60 + seconds
    return days * 86400 + minutes * 60 + seconds

laps_raw = spark.table("formula1_eda_race.laps")

laps_parsed = (
    laps_raw
    .withColumn("total_laptime_secs", timedelta_str_to_seconds("LapTime"))
    .withColumn("pit_in_secs",  timedelta_str_to_seconds("PitInTime"))
    .withColumn("pit_out_secs", timedelta_str_to_seconds("PitOutTime"))
)

# Persist the parsed version back to the database so downstream queries can just
# read formula1.laps directly (mirrors what spark_checker.ipynb did).
laps_parsed.write.mode("overwrite").saveAsTable("formula1_eda_race.laps_parsed")
laps_parsed.select("Driver", "LapNumber", "LapTime", "total_laptime_secs",
                    "pit_in_secs", "pit_out_secs").show(10)


+------+---------+--------------------+------------------+-----------+------------+
|Driver|LapNumber|             LapTime|total_laptime_secs|pit_in_secs|pit_out_secs|
+------+---------+--------------------+------------------+-----------+------------+
|   GAS|      1.0|0 days 00:01:22.8...| 82.82300000000001|       NULL|        NULL|
|   GAS|      2.0|0 days 00:01:33.5...|            93.553|       NULL|        NULL|
|   GAS|      3.0|0 days 00:01:31.4...| 91.40899999999999|       NULL|        NULL|
|   GAS|      4.0|0 days 00:01:29.5...|            89.513|       NULL|        NULL|
|   GAS|      5.0|0 days 00:01:32.3...|            92.311|       NULL|        NULL|
|   GAS|      6.0|0 days 00:01:21.4...| 81.47200000000001|       NULL|        NULL|
|   GAS|      7.0|0 days 00:00:59.5...|            59.592|       NULL|        NULL|
|   GAS|      8.0|0 days 00:00:59.3...|            59.323|       NULL|        NULL|
|   GAS|      9.0|0 days 00:00:58.9...|            58.997|       NULL|      

In [11]:
# ── Split results into race and qualifying classifications ────────────────────
# type_of_race was parsed from the folder name during ingestion ('R' vs 'Q').
# Two events (year, session_number) can share the same DriverNumber, so that's
# the join key used throughout this section.
results = spark.table("formula1_eda_race.results")

race_results = results.filter(col("type_of_race") == "R")

quali_results = (
    results
    .filter(col("type_of_race") == "Q")
    .select("year", "session_number", "DriverNumber",
            col("Position").alias("quali_pos"))
)
quali_results = quali_results.withColumn("quali_pos", col("quali_pos").cast("int"))




# ── Overtakes: count every lap-to-lap position improvement during the race ────
# Same logic as f1.ipynb's cell 4, generalized to (year, session_number) instead
# of (year, round) since that's how events are keyed in the database.
race_laps = (
    laps_parsed
    .filter(col("type_of_race") == "R")
    .withColumn("Position", col("Position").cast("int"))
)

overtake_window = Window.partitionBy("year", "session_number", "DriverNumber").orderBy("LapNumber")

overtakes_spark = (
    race_laps
    .withColumn("prev_position", F.lag("Position", 1).over(overtake_window))
    .withColumn(
        "overtake_this_lap",
        when(col("prev_position").isNotNull() & (col("Position") < col("prev_position")), 1).otherwise(0)
    )
    .groupBy("year", "session_number", "DriverNumber")
    .agg(F.sum("overtake_this_lap").alias("overtakes_made"))
)

overtakes_spark.orderBy(F.desc("overtakes_made")).show(5)


# ── Best pit-stop duration per driver per race ─────────────────────────────────
# Same in-lap / out-lap pairing logic as f1.ipynb's cell 5: an in-lap (where the
# driver entered the pits) is paired with the *next* out-lap (where they left)
# for that same driver in that same race, and the duration is out_time - in_time.
# best_pit_sec is deliberately left null (not filled with 0) when a driver never
# pitted or retired mid-pit-entry — null correctly means "no valid stop to report."
in_laps = (
    race_laps
    .filter(col("pit_in_secs").isNotNull())
    .select("year", "session_number", "DriverNumber", "LapNumber",
            col("pit_in_secs").alias("pit_in_time"))
)

out_laps = (
    race_laps
    .filter(col("pit_out_secs").isNotNull())
    .select("year", "session_number", "DriverNumber", "LapNumber",
            col("pit_out_secs").alias("pit_out_time"))
)

paired_pits = (
    in_laps.alias("i")
    .join(
        out_laps.alias("o"),
        (col("i.year") == col("o.year")) &
        (col("i.session_number") == col("o.session_number")) &
        (col("i.DriverNumber") == col("o.DriverNumber")) &
        (col("o.pit_out_time") > col("i.pit_in_time")),
        how="inner",
    )
    .withColumn("pit_duration_sec", col("o.pit_out_time") - col("i.pit_in_time"))
)

best_pit_spark = (
    paired_pits
    .groupBy(col("i.year").alias("year"), col("i.session_number").alias("session_number"),
              col("i.DriverNumber").alias("DriverNumber"))
    .agg(F.min("pit_duration_sec").alias("best_pit_sec"))
)

# ── Fastest lap flag: whoever set the quickest race lap in each event ─────────
fastest_lap_window = Window.partitionBy("year", "session_number").orderBy("total_laptime_secs")

fastest_lap_spark = (
    race_laps
    .filter(col("total_laptime_secs").isNotNull())
    .withColumn("rnk", F.rank().over(fastest_lap_window))
    .filter(col("rnk") == 1)
    .select("year", "session_number", "DriverNumber")
    .withColumn("fastest_lap_flag", F.lit(1))
)


# ── Point value constants ──────────────────────────────────────────────────────
### hannah note: took some liberties and assigned points here
# Standard F1-style points-paying positions, same tables as f1.ipynb.
RACE_PTS  = {1: 25, 2: 18, 3: 15, 4: 12, 5: 10, 6: 8, 7: 6, 8: 4, 9: 2, 10: 1}
QUALI_PTS = {1: 10, 2: 9,  3: 8,  4: 7,  5: 6,  6: 5, 7: 4, 8: 3, 9: 2, 10: 1}

# NOTE: f1.ipynb's source cells were cut off before showing the exact multiplier
# used for pts_overtake and pts_fastest. These are set to a sensible default
# (1 point per overtake, 5 points for fastest lap) — change PTS_PER_OVERTAKE /
# FASTEST_LAP_BONUS below.
PTS_PER_OVERTAKE  = 1
FASTEST_LAP_BONUS = 5

def position_to_points(col_name, pts_dict):
    """Map a position column to fantasy points using a lookup dict; unranked = 0."""
    expr = None
    for pos, pts in pts_dict.items():
        expr = when(col(col_name) == pos, pts) if expr is None else expr.when(col(col_name) == pos, pts)
    return expr.otherwise(0)




+----+--------------+------------+--------------+
|year|session_number|DriverNumber|overtakes_made|
+----+--------------+------------+--------------+
|2022|             1|          24|            22|
|2023|             7|          16|            21|
|2023|            17|          11|            21|
|2022|            11|          14|            20|
|2023|            17|          63|            20|
+----+--------------+------------+--------------+
only showing top 5 rows


In [35]:
# ── Per-race lap-based features (pace, consistency, pit stops) ────────────────
race_laps_clean = race_laps.filter(col("total_laptime_secs") > 0)

session_fastest = (
    race_laps_clean
    .groupBy("year", "session_number")
    .agg(F.min("total_laptime_secs").alias("session_fastest_laptime"))
)

lap_features = (
    race_laps_clean
    .groupBy("year", "session_number", "DriverNumber")
    .agg(
        F.mean("total_laptime_secs").alias("avg_laptime_secs"),
        F.stddev("total_laptime_secs").alias("laptime_std"),
        F.count("*").alias("laps_completed"),
        F.mean("Position").alias("avg_race_position"),
        F.sum(when(col("pit_out_secs").isNotNull(), 1).otherwise(0)).alias("pit_stop_count"),
    )
    .join(session_fastest, on=["year", "session_number"], how="left")
    # pace_gap_pct: how far off the fastest lap of the race this driver's average lap was
    .withColumn("pace_gap_pct",
        (col("avg_laptime_secs") - col("session_fastest_laptime")) / col("session_fastest_laptime") * 100)
    # laptime_cv: coefficient of variation — lower means more consistent lap times
    .withColumn("laptime_cv", col("laptime_std") / col("avg_laptime_secs") * 100))

In [39]:
# ── Join everything onto the race results and compute the fantasy formula ─────
master_spark = (
    race_results
    .join(quali_results,      on=["year", "session_number", "DriverNumber"], how="left")
    .join(overtakes_spark,    on=["year", "session_number", "DriverNumber"], how="left")
    .join(best_pit_spark,     on=["year", "session_number", "DriverNumber"], how="left")
    .join(fastest_lap_spark,  on=["year", "session_number", "DriverNumber"], how="left")
    .join(weather_averages, on = ["year", "session_number"], how="left")
    .join(flag_counts, on = ["year", "session_number"], how="left")
    .join(race_name, on = ["year", "session_number"], how="left")
    .join(lap_features, on = ["year", "session_number"], how="left")
    .fillna({"overtakes_made": 0, "fastest_lap_flag": 0})
    # best_pit_sec is intentionally NOT filled — null means "no valid pit stop", not zero seconds
)

master_spark = (
    master_spark
    .withColumn("Position",     col("Position").cast("int"))
    .withColumn("GridPosition", col("GridPosition").cast("int"))
    .withColumn("pts_race",         position_to_points("Position", RACE_PTS))
    .withColumn("pts_quali",        position_to_points("quali_pos", QUALI_PTS))
    .withColumn("positions_gained", col("GridPosition") - col("Position"))
    .withColumn("pts_pos",          col("positions_gained") * 2)
    .withColumn("pts_overtake",     col("overtakes_made") * PTS_PER_OVERTAKE)
    .withColumn("pts_fastest",      col("fastest_lap_flag") * FASTEST_LAP_BONUS)
    .withColumn(
        "fantasy_pts_total",
        col("pts_race") + col("pts_quali") + col("pts_pos") + col("pts_overtake") + col("pts_fastest")
    )
)

master_spark.select(
    "year", "session_number", "results.DriverNumber", "Abbreviation", "TeamName", "Position", "GridPosition", "Status", "best_pit_sec",
    "fastest_lap_flag", "pts_race", "pts_quali", "positions_gained", "pts_pos", "pts_overtake", "pts_fastest", "fantasy_pts_total",
    "rain", "active_flag_count", "Location", "avg_laptime_secs", "laptime_std", "laps_completed", "avg_race_position", "pit_stop_count",
    "session_fastest_laptime", "pace_gap_pct", "laptime_cv"
).show(10)
master_spark.show(10)

# Persist as a queryable table too, mirroring the "write master to the database" step
# f1.ipynb did with parquet — here it goes into the same Spark SQL database instead.
(
    master_spark
    .repartition("year")
    .write.mode("overwrite")
    .partitionBy("year")
    .saveAsTable("formula1_eda_race.master")
)

+----+--------------+------------+------------+--------+--------+------------+--------+------------+----------------+--------+---------+----------------+-------+------------+-----------+-----------------+-----+-----------------+--------+-----------------+------------------+--------------+------------------+--------------+-----------------------+------------------+------------------+
|year|session_number|DriverNumber|Abbreviation|TeamName|Position|GridPosition|  Status|best_pit_sec|fastest_lap_flag|pts_race|pts_quali|positions_gained|pts_pos|pts_overtake|pts_fastest|fantasy_pts_total| rain|active_flag_count|Location| avg_laptime_secs|       laptime_std|laps_completed| avg_race_position|pit_stop_count|session_fastest_laptime|      pace_gap_pct|        laptime_cv|
+----+--------------+------------+------------+--------+--------+------------+--------+------------+----------------+--------+---------+----------------+-------+------------+-----------+-----------------+-----+-----------------+

+----+--------------+------------+-------------+------------+--------+--------+---------+-------+---------+--------+---------------+--------------------+-----------+--------+------------------+------------+----+----+----+--------------------+--------+------+----+--------------------+------------+---------+--------------+-----------------+----------------+------------------+------------------+-----------------+-----+-----------------+------------------+------------------+-----------------+--------------------+------------+-----------+---------+------------+-----------------+------------------+--------------+------------------+--------------+-----------------------+------------------+------------------+--------+---------+----------------+-------+------------+-----------+-----------------+
|year|session_number|DriverNumber|BroadcastName|Abbreviation|DriverId|TeamName|TeamColor| TeamId|FirstName|LastName|       FullName|         HeadshotUrl|CountryCode|Position|ClassifiedPosition|GridPosit

AnalysisException: [COLUMN_ALREADY_EXISTS] The column `drivernumber` already exists. Choose another name or rename the existing column. SQLSTATE: 42711